<a href="https://colab.research.google.com/github/Ganesh-Navadeep/Predictive-Analytics/blob/House-price-prediction/Sudent_join_predictoion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = pd.read_csv("student_admission_dataset.csv")

# Encode target variable
le = LabelEncoder()
df['Admission_Status'] = le.fit_transform(df['Admission_Status'])  # Accepted=1, Rejected=0

# Features and target
X = df[['GPA', 'SAT_Score', 'Extracurricular_Activities']]
y = df['Admission_Status']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model training
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
print("\nModel Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=le.classes_))

# --- User input section ---
def get_user_input():
    print("\nEnter student details:")
    gpa = float(input("GPA (0.0 - 4.0): "))
    sat = int(input("SAT Score: "))
    activities = int(input("Number of extracurricular activities: "))
    return pd.DataFrame([[gpa, sat, activities]], columns=['GPA', 'SAT_Score', 'Extracurricular_Activities'])

# Predict user input
user_df = get_user_input()
prediction = model.predict(user_df)[0]

print("\nPredicted Admission Status:", le.inverse_transform([prediction])[0])



Model Accuracy: 0.28
Classification Report:
               precision    recall  f1-score   support

    Accepted       0.25      0.29      0.27        14
    Rejected       0.29      0.33      0.31        18
  Waitlisted       0.31      0.22      0.26        18

    accuracy                           0.28        50
   macro avg       0.28      0.28      0.28        50
weighted avg       0.28      0.28      0.28        50


Enter student details:
GPA (0.0 - 4.0): 3.9
SAT Score: 220
Number of extracurricular activities: 4

Predicted Admission Status: Waitlisted


In [4]:
df

,GPA,SAT_Score,Extracurricular_Activities,Admission_Status
0,3.46,1223,8,1
1,2.54,974,8,1
2,2.91,909,9,1
3,2.83,1369,5,0
4,3.60,1536,7,0
...,...,...,...,...
245,3.57,1024,3,1
246,2.86,1367,1,2
247,3.09,1036,3,2
248,3.51,1375,5,2


In [7]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf

# 1. Fetch historical ETH data from CoinGecko
def fetch_eth_history(days=90):
    url = f"https://api.coingecko.com/api/v3/coins/ethereum/market_chart"
    params = {"vs_currency": "usd", "days": days}
    data = requests.get(url, params=params).json()
    prices = data['prices']  # list of [timestamp, price]
    df = pd.DataFrame(prices, columns=['timestamp', 'price'])
    df['ds'] = pd.to_datetime(df['timestamp'], unit='ms')
    df = df[['ds', 'price']].set_index('ds')
    return df

# 2. Prepare data for LSTM
def prepare_lstm_data(df, look_back=60):
    scaler = MinMaxScaler(feature_range=(0,1))
    scaled = scaler.fit_transform(df)
    X, y = [], []
    for i in range(look_back, len(scaled)):
        X.append(scaled[i-look_back:i, 0])
        y.append(scaled[i, 0])
    return np.array(X), np.array(y), scaler

# 3. Build and train the LSTM model
def train_lstm(X, y, epochs=10, batch_size=16):
    X = X.reshape((X.shape[0], X.shape[1], 1))
    model = tf.keras.Sequential([
        tf.keras.layers.LSTM(50, return_sequences=True, input_shape=(X.shape[1], 1)),
        tf.keras.layers.LSTM(50),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=1)
    return model

# 4. Predict future prices
def predict_future(model, last_data, scaler, future_steps):
    arr = last_data.copy()
    preds = []
    for _ in range(future_steps):
        inp = arr[-look_back:].reshape(1, look_back, 1)
        pred = model.predict(inp)[0,0]
        preds.append(pred)
        arr = np.append(arr, pred)
    preds = scaler.inverse_transform(np.array(preds).reshape(-1,1)).flatten()
    return preds

# === MAIN ===
if __name__ == "__main__":
    # User inputs
    look_back = int(input("Enter look-back window size (e.g., 60): "))
    future_steps = int(input("Predict how many hours into the future? "))

    # Data fetching and prep
    df = fetch_eth_history(days=30)  # last 30 days
    X, y, scaler = prepare_lstm_data(df, look_back)

    # Train-test split
    split_idx = int(len(X)*0.8)
    X_train, y_train = X[:split_idx], y[:split_idx]
    X_test, y_test = X[split_idx:], y[split_idx:]

    model = train_lstm(X_train, y_train, epochs=20, batch_size=32)

    # Optionally evaluate performance
    test_X = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))
    y_pred = model.predict(test_X)
    y_pred = scaler.inverse_transform(y_pred)
    y_true = scaler.inverse_transform(y_test.reshape(-1,1))
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    print(f"Test RMSE: ${rmse:.2f}")

    # Forecast future prices
    last_seq = scaler.transform(df.values)[-look_back:].flatten()
    future_prices = predict_future(model, last_seq, scaler, future_steps)

    # Output results
    for i, price in enumerate(future_prices, start=1):
        ts = datetime.now() + pd.Timedelta(hours=i)
        print(f"{ts.strftime('%Y-%m-%d %H:%M')}: ${price:.2f}")



Enter look-back window size (e.g., 60): 120
Predict how many hours into the future? 96
Epoch 1/20


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


15/15 ━━━━━━━━━━━━━━━━━━━━ 5s 82ms/step - loss: 0.1569
Epoch 2/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step - loss: 0.0093
Epoch 3/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 115ms/step - loss: 0.0074
Epoch 4/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 82ms/step - loss: 0.0050
Epoch 5/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step - loss: 0.0046
Epoch 6/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step - loss: 0.0041
Epoch 7/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 79ms/step - loss: 0.0039
Epoch 8/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step - loss: 0.0034
Epoch 9/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step - loss: 0.0030
Epoch 10/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 85ms/step - loss: 0.0032
Epoch 11/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 124ms/step - loss: 0.0029
Epoch 12/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 82ms/step - loss: 0.0029
Epoch 13/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - loss: 0.0028
Epoch 14/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 80ms/step - loss: 0.0031
Epoch 15/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 78ms/step - loss: 0.0026
Epoch 16/20
1

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━

In [8]:
pip install pandas numpy requests scikit-learn tensorflow


In [9]:
import requests
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, timedelta
import tensorflow as tf

# Step 1: Fetch historical ETH/USD prices (last 90 days hourly)
def fetch_eth_data(days=90):
    url = "https://api.coingecko.com/api/v3/coins/ethereum/market_chart"
    params = {"vs_currency": "usd", "days": days}
    res = requests.get(url, params=params).json()
    prices = res["prices"]  # [timestamp, price]
    df = pd.DataFrame(prices, columns=["timestamp", "price"])
    df["ds"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df.set_index("ds")[["price"]]
    df = df.asfreq("H").interpolate()  # ensure hourly frequency
    return df

# Step 2: Prepare dataset for LSTM
def prepare_data(series, look_back=72):
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(series)
    X, y = [], []
    for i in range(look_back, len(scaled)):
        X.append(scaled[i-look_back:i, 0])
        y.append(scaled[i, 0])
    return np.array(X), np.array(y), scaler

# Step 3: Build and train LSTM model
def train_model(X, y, epochs=10, batch_size=32):
    X = X.reshape(X.shape[0], X.shape[1], 1)
    model = tf.keras.Sequential([
        tf.keras.layers.LSTM(64, return_sequences=True, input_shape=(X.shape[1], 1)),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dense(1)
    ])
    model.compile(optimizer="adam", loss="mean_squared_error")
    model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=1)
    return model

# Step 4: Forecast next n steps
def forecast_next(model, last_sequence, steps, scaler):
    predictions = []
    curr_seq = last_sequence.copy()
    for _ in range(steps):
        inp = curr_seq[-look_back:].reshape(1, look_back, 1)
        pred = model.predict(inp, verbose=0)[0, 0]
        predictions.append(pred)
        curr_seq = np.append(curr_seq, pred)
    return scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()

# Step 5: Put it all together
if __name__ == "__main__":
    look_back = 72
    future_hours = 120

    print("Fetching ETH/USD data...")
    df = fetch_eth_data(days=90)

    print("Preparing data...")
    X, y, scaler = prepare_data(df.values, look_back=look_back)
    model = train_model(X, y, epochs=15)

    print(f"\nPredicting next {future_hours} hours...")
    last_seq = scaler.transform(df.values)[-look_back:].flatten()
    future_prices = forecast_next(model, last_seq, future_hours, scaler)

    # Display predictions with timestamps
    last_time = df.index[-1]
    future_times = [last_time + timedelta(hours=i+1) for i in range(future_hours)]

    for t, p in zip(future_times, future_prices):
        print(f"{t.strftime('%Y-%m-%d %H:%M')} → ${p:.2f}")


Fetching ETH/USD data...


/tmp/ipython-input-9-2074558840.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df = df.asfreq("H").interpolate()  # ensure hourly frequency
/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Preparing data...
Epoch 1/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 20s 110ms/step - loss: 0.0000e+00
Epoch 2/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - loss: 0.0000e+00
Epoch 3/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 6s 67ms/step - loss: 0.0000e+00
Epoch 4/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 4s 54ms/step - loss: 0.0000e+00
Epoch 5/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 6s 62ms/step - loss: 0.0000e+00
Epoch 6/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 5s 54ms/step - loss: 0.0000e+00
Epoch 7/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - loss: 0.0000e+00
Epoch 8/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 6s 65ms/step - loss: 0.0000e+00
Epoch 9/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 4s 53ms/step - loss: 0.0000e+00
Epoch 10/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step - loss: 0.0000e+00
Epoch 11/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 5s 70ms/step - loss: 0.0000e+00
Epoch 12/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - loss: 0.0000e+00
Epoch 13/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - loss: 0.0000e+00
Epoch 14/15
66/66 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - loss: 0.0000

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Load dataset
df = pd.read_csv("creditcard.csv")

# Features and target
# Load dataset
df = pd.read_csv("creditcard.csv")

# Drop rows where 'Class' is NaN
df = df[df["Class"].notna()]

# Drop rows where other features are NaN (just in case)
df = df.dropna()

# Features and target
X = df.drop("Class", axis=1)
y = df["Class"]


# Scale 'Amount' and 'Time' (other features are already PCA-transformed)
scaler = StandardScaler()
X[["Time", "Amount"]] = scaler.fit_transform(X[["Time", "Amount"]])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
print("\nModel Evaluation:")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))

# --- User Input Section ---6
def get_user_input():
    print("\nEnter transaction details:")
    input_data = {}
    input_data["Time"] = float(input("Time (in seconds since first transaction): "))
    for i in range(1, 29):
        input_data[f"V{i}"] = float(input(f"V{i}: "))
    input_data["Amount"] = float(input("Transaction Amount: "))

    # Convert to DataFrame
    input_df = pd.DataFrame([input_data])
    input_df[["Time", "Amount"]] = scaler.transform(input_df[["Time", "Amount"]])
    return input_df

# Get input and make prediction
user_input_df = get_user_input()
prediction = model.predict(user_input_df)[0]
print("\n🔍 Prediction Result:")
print("🚨 Fraudulent Transaction!" if prediction == 1 else "✅ Legitimate Transaction.")



Model Evaluation:
[[56861     3]
 [   25    73]]
              precision    recall  f1-score   support

           0     0.9996    0.9999    0.9998     56864
           1     0.9605    0.7449    0.8391        98

    accuracy                         0.9995     56962
   macro avg     0.9800    0.8724    0.9194     56962
weighted avg     0.9995    0.9995    0.9995     56962


Enter transaction details:
Time (in seconds since first transaction): 20
V1: 2000
V2: 2100
V3: 5000
V4: 300
V5: 400
V6: 4000
V7: 9000
V8: 1230
V9: 215
V10: 1623
V11: 6632.2
V12: 111.11
V13: 2164236
V14: 21636
V15: 333
V16: 31949
V17: 95496
V18: 956
V19: 66
V20: 66956
V21: 51466
V22: 446
V23: 64596
V24: 65496
V25: 69
V26: 4596
V27: 6959546
V28: 694965
Transaction Amount: 5000

🔍 Prediction Result:
✅ Legitimate Transaction.


In [22]:
# --- Dynamic Input Function ---
def get_user_profile():
    print("\n🧾 Enter New Customer Profile Details:")
    user_data = {}
    user_data["Gender"] = input("Gender (Male/Female): ").title()
    user_data["Age"] = int(input("Age: "))
    user_data["Category"] = input("Product Category: ").title()
    user_data["Shopping_Mall"] = input("Shopping Mall: ").title()
    user_data["Payment_Method"] = input("Payment Method (Cash/Credit card/Debit card/eWallet): ").title()
    user_data["Spending_Score_(1-100)"] = int(input("Spending Score (1-100): "))
    user_data["Purchase_Amount_(USD)"] = float(input("Purchase Amount ($): "))
    user_data["Income"] = input("Income (e.g., Low, Medium, High): ").title()
    return pd.DataFrame([user_data])

# --- Comparison Function ---
def compare_user_to_data(user_df):
    print("\n🔍 Analysis of Entered Profile:")

    gender_avg = df[df["Gender"] == user_df.iloc[0]["Gender"]]["Purchase_Amount_(USD)"].mean()
    print(f"🧑 Average spending for {user_df.iloc[0]['Gender']} customers: ${gender_avg:.2f}")

    category_popularity = df["Category"].value_counts(normalize=True) * 100
    category = user_df.iloc[0]["Category"]
    if category in category_popularity:
        print(f"🛒 {category} makes up {category_popularity[category]:.2f}% of purchases.")
    else:
        print(f"🛒 {category} is a rare or new product category.")

    print("\n📈 Spending Comparison:")
    plt.figure(figsize=(6, 4))
    sns.histplot(df["Purchase_Amount_(USD)"], bins=30, kde=True, color='skyblue', label="Existing Customers")
    plt.axvline(user_df.iloc[0]["Purchase_Amount_(USD)"], color='red', linestyle='--', label="Your Input")
    plt.legend()
    plt.title("Purchase Amount Distribution")
    plt.xlabel("Purchase Amount (USD)")
    plt.tight_layout()
    plt.show()
